In [21]:
import pandas as pd
from functools import reduce
import glob
import os
import re

In [23]:
# import tables
# import spn_pbp_amr
resistance = pd.read_csv("../data_file/pathogenwatch_output20251221/spn_pbp_amr_use_this.csv")
# import serotype
serotype = pd.read_csv("../data_file/pathogenwatch_output20251221/serotype_use_this.csv")
# import mlst-pubmlst
mlst = pd.read_csv("../data_file/pathogenwatch_output20251221/mlst-pubmlst.csv")
# import metadata
metadata = pd.read_excel("../data_file/processed_clinical_data20260106_1.xlsx") # preprocessed manually
breakpoint = pd.read_excel("../data_file/Standard_breakpoint_table_for_SPN.xlsx") # not in use

In [25]:
# process metatable, for every MIC, consult the breakpoint table to obtain S/I/R to use for heatmap (done manually)
# for each sample, only keep accessionNum, fasta_id, antibiotics S/I/R
meta_col = ["accessionNum", "Genome Name", "Clinical Serotype", "Penicillin", "Penicillin_original", "pcn_sens_np",
            "Ceftriaxone", "Erythromycin", "Clindamycin", 
            "Vancomycin", "Tetracycline", "Levofloxacin", "NP swap/ear swap", "ageInMonths"]

select_meta = metadata[meta_col]

In [27]:
select_meta

,accessionNum,Genome Name,Clinical Serotype,Penicillin,Penicillin_original,pcn_sens_np,Ceftriaxone,Erythromycin,Clindamycin,Vancomycin,Tetracycline,Levofloxacin,NP swap/ear swap,ageInMonths
0,W17143487,2_S1_SPN,23B,S,S,Susceptible,S,S,S,S,S,S,NP,31
1,F16380623,3_S6_SPN,35B,I,I,Intermediate,S,R,S,S,S,S,NP,30
2,W17653277,4_S2_SPN,35B,I,I,Intermediate,S,R,S,S,S,S,NP,11
3,M17701635,5_S15_SPN,35B & 35D,S,I,Intermediate,S,R,S,S,S,S,NP,18
4,T18271326,6_S19_SPN,3,S,S,Susceptible,S,S,S,S,S,S,NP,29
5,M18434558,7_S13_SPN,17F,S,S,Susceptible,S,S,S,S,S,S,NP,12
6,F17654457,8_S30_SPN,17F,S,S,Susceptible,S,S,S,S,S,S,NP,6
7,F17896493,9_S33_SPN,17F,S,S,Susceptible,S,S,S,S,S,S,NP,30
8,F18239423,10_S22_SPN,35B,I,I,Intermediate,S,R,S,S,S,S,NP,22
9,T19376701,11_S7_SPN,3,S,S,Susceptible,S,R,R,S,R,S,NP,6


In [29]:
# process resistance table, rename to corresponding antibiotic
# for each sample, only keep Genome Name, antibiotic S/I/R
resist_col = ["Genome Name", 
              "amx", "croNonMeningitis", 
              "ctxNonMeningitis", "cxm", 
              "mem", "penNonMeningitis"]
select_resist = resistance[resist_col]

In [31]:
# process serotype table, keep only Genome Name and computed serotype
sero_col = ["Genome Name", "Serotype"]
select_patho_sero = serotype[sero_col]

In [33]:
# process mlst table, keep only Genome Name and ST
mlst_col = ["Genome Name", "ST"]
select_mlst = mlst[mlst_col]

In [35]:
# merge all tables based on Genome Name (fasta_id)
dfs = [
    select_meta, select_resist, select_patho_sero, select_mlst
]

merged = reduce(
    lambda left, right: pd.merge(left, right, on="Genome Name", how="left"),
    dfs
)

In [37]:
merged

,accessionNum,Genome Name,Clinical Serotype,Penicillin,Penicillin_original,pcn_sens_np,Ceftriaxone,Erythromycin,Clindamycin,Vancomycin,...,NP swap/ear swap,ageInMonths,amx,croNonMeningitis,ctxNonMeningitis,cxm,mem,penNonMeningitis,Serotype,ST
0,W17143487,2_S1_SPN,23B,S,S,Susceptible,S,S,S,S,...,NP,31,S,S,S,S,S,S,23B,1262
1,F16380623,3_S6_SPN,35B,I,I,Intermediate,S,R,S,S,...,NP,30,I,S,S,R,I,R,35B,558
2,W17653277,4_S2_SPN,35B,I,I,Intermediate,S,R,S,S,...,NP,11,NF,NaN,NaN,NaN,NF,NF,35B,37ba893024bc976f4b4de93628e4778e9a4726ca
3,M17701635,5_S15_SPN,35B & 35D,S,I,Intermediate,S,R,S,S,...,NP,18,I,S,S,R,I,R,35B,dbf4e9531e53ffa309916adceab4f755649143ce
4,T18271326,6_S19_SPN,3,S,S,Susceptible,S,S,S,S,...,NP,29,S,S,S,S,S,S,03,180
5,M18434558,7_S13_SPN,17F,S,S,Susceptible,S,S,S,S,...,NP,12,S,S,S,S,S,S,17F,392
6,F17654457,8_S30_SPN,17F,S,S,Susceptible,S,S,S,S,...,NP,6,S,S,S,S,S,S,17F,392
7,F17896493,9_S33_SPN,17F,S,S,Susceptible,S,S,S,S,...,NP,30,S,S,S,S,S,S,17F,392
8,F18239423,10_S22_SPN,35B,I,I,Intermediate,S,R,S,S,...,NP,22,I,S,S,R,I,R,35B,156
9,T19376701,11_S7_SPN,3,S,S,Susceptible,S,R,R,S,...,NP,6,S,S,S,S,S,S,03,180


In [39]:
merged.to_csv("merged_table_20260106.csv", index=False)
